# 📘 06 - Function Pipelines

Everything in this module leads to one idea:

> **Data comes in. Data gets transformed. The result of one step becomes the input to the next.**

That idea is a **pipeline**, and it's what most data science code actually does. Pandas, which is coming up soon, is basically a big collection of well-built pipeline steps.

## ✅ Learning Goals
- Look at one long block of code and find where the separate functions should go
- Refactor a "blob" into small functions that pass data between each other
- Draw a pipeline: what goes into each step, and what comes out
- Reuse the same pipeline on different data without changing it

In [ ]:
# A tiny helper to test your functions. Run this cell first!
def check(actual, expected):
    if actual == expected:
        print("✅", actual)
    else:
        print("❌ got", actual, "but expected", expected)

## 🧭 The Mantra

For any problem, ask these questions in order:

```text
1. Where is my data?
2. What shape is my data?            (list? list of dicts? dictionary?)
3. What needs to happen to it?
4. Can that operation become a function?
5. What goes in?
6. What comes out?
```

---

## 🧩 The Problem

```python
numbers = [12, -5, 8, 22, -9, 15, 31]
```

> **Remove the negatives, calculate the average, and find all the numbers above that average.**

### Step 1 — The Blob

Most people's first attempt looks like this: one big block of code that does everything.

🔮 **Predict** what it prints. (Trace `clean`, then `avg`, then `result`.)

In [ ]:
numbers = [12, -5, 8, 22, -9, 15, 31]

clean = []
for n in numbers:
    if n >= 0:
        clean.append(n)

total = 0
for n in clean:
    total += n
avg = total / len(clean)

result = []
for n in clean:
    if n > avg:
        result.append(n)

print(result)

✍️ **My prediction:** _(write it here BEFORE you run the next cell)_

The blob works, and working is a fine place to start. But:

- To understand it, you have to read all 15 lines.
- If you need "the average of the non-negatives" somewhere else, you have to copy lines 3–11.
- You can't test the "remove negatives" part by itself.

### Step 2 — Find the Seams

Look at the blob again. Blank lines divide it into **three parts**. Each part:

- uses a value that the part before it produced
- produces a value that the next part uses

✍️ For each part, name the pattern (Accumulate / Count / Filter), and write down what goes **in** and what comes **out**.

| Part | Pattern | In | Out | Good function name |
|------|---------|----|-----|--------------------|
| 1 | | | | |
| 2 | | | | |
| 3 | | | | |

✍️ **My table:**

### Step 3 — Refactor Into Functions

Each part becomes a function. You've already written most of them in Notebook 05. Here they are, with contracts:

In [ ]:
def remove_negatives(numbers):
    """IN: list of numbers   OUT: new list   DOES: drops numbers below 0"""
    result = []
    for n in numbers:
        if n >= 0:
            result.append(n)
    return result


def average(numbers):
    """IN: non-empty list of numbers   OUT: float   DOES: mean"""
    return sum(numbers) / len(numbers)


def above(numbers, cutoff):
    """IN: list of numbers, a number   OUT: new list   DOES: keeps numbers > cutoff"""
    result = []
    for n in numbers:
        if n > cutoff:
            result.append(n)
    return result

### Step 4 — Connect Them

In [ ]:
numbers = [12, -5, 8, 22, -9, 15, 31]

clean  = remove_negatives(numbers)
avg    = average(clean)
result = above(clean, avg)

print(result)

Same answer as the blob, but now **each line reads like a sentence**. Here's how the data flows:

```text
        numbers
           │
           ▼
 ┌──────────────────┐
 │ remove_negatives │
 └──────────────────┘
           │
           ▼
         clean ─────────────────┐
           │                    │
           ▼                    │
     ┌───────────┐              │
     │  average  │              │
     └───────────┘              │
           │                    │
           ▼                    │
          avg                   │
           │                    │
           └────────┬───────────┘
                    ▼
              ┌───────────┐
              │   above   │
              └───────────┘
                    │
                    ▼
                  result
```

Notice that `clean` is used **twice**: once to compute the average and again to filter. Pipelines don't have to be a straight line.

| Step | In (shape) | Out (shape) |
|------|------------|-------------|
| `remove_negatives` | list | list |
| `average` | list | number |
| `above` | list + number | list |

🔮 **Check the connections:** the output shape of each step has to match the input shape of the next one. Where in the diagram would a `print`-instead-of-`return` bug break the pipeline?

---

## ♻️ The Payoff: Same Pipeline, New Data

We didn't write the functions for these particular seven numbers. **We didn't change a single function** below:

In [ ]:
temps = [31, -4, 45, 28, -10, 52, 60, 30, 71]

clean  = remove_negatives(temps)
avg    = average(clean)
print("above-average temps:", above(clean, avg))

Since the three steps always go together, you can even put the **whole pipeline** in a function of its own. That's a function made of functions:

In [ ]:
def above_average_nonnegatives(numbers):
    clean = remove_negatives(numbers)
    return above(clean, average(clean))

check(above_average_nonnegatives([12, -5, 8, 22, -9, 15, 31]), [22, 31])

---

## 🧑‍🎓 A Pipeline on Real-Looking Data

Now use the shape you'll see most often: a **list of dictionaries** (rows in a table).

> **Which students scored above the class average?**

Run through the mantra: the data is in `students`. Its shape is a list of dicts. We need to get the grades out, average them, keep the students above that average, and pull out their names.

In [ ]:
students = [
    {"name": "Alice", "grade": 91},
    {"name": "Bob",   "grade": 84},
    {"name": "Carol", "grade": 95},
    {"name": "Dev",   "grade": 62},
    {"name": "Eli",   "grade": 78}
]

def grades_of(rows):
    """IN: list of student dicts   OUT: list of numbers"""
    result = []
    for r in rows:
        result.append(r["grade"])
    return result

def students_above(rows, cutoff):
    """IN: list of student dicts, a number   OUT: list of student dicts"""
    # Your code here (a Filter on r["grade"])
    pass

def names_of(rows):
    """IN: list of student dicts   OUT: list of names"""
    # Your code here
    pass

In [ ]:
# The pipeline. Predict the output before you run it!
grades = grades_of(students)
avg    = average(grades)
top    = students_above(students, avg)
print(names_of(top))                 # expected: ['Alice', 'Bob', 'Carol']

```text
 students ──► grades_of ──► grades ──► average ──► avg
    │                                               │
    └──────────────► students_above ◄───────────────┘
                           │
                           ▼
                          top ──► names_of ──► ['Alice', 'Bob', 'Carol']
```

When you get to Pandas, this whole thing will look something like:

```python
df[df["grade"] > df["grade"].mean()]["name"]
```

It's the same pipeline written in one line, and you'll understand it because you built each piece yourself.

---

## 🤝 Team Activity: Build a Pipeline Together

Each team writes **one** function. Nobody writes the whole thing.

1. As a class, agree on the **contracts** (in / out / does) for every step first.
2. Each team writes and tests its function.
3. Paste everyone's functions into one notebook and connect them.

If the contracts were clear, the pipeline works the first time you connect it. If not, you'll quickly find out which contract was vague.

---

## ✏️ Your Turn — A Word Pipeline

> **What is the most common word in a sentence, ignoring upper/lower case?**

The contracts are written for you. Implement each function, then connect them.

In [ ]:
text = "The cat and the hat and THE bat"

def to_words(sentence):
    """IN: a string   OUT: list of lowercase words   (hint: .lower() and .split())"""
    pass

def frequency(values):
    """IN: list   OUT: dict {value: count}   (you wrote this in Notebook 05)"""
    pass

def most_common(counts):
    """IN: dict {value: count}   OUT: the key with the largest count (Best So Far)"""
    pass

# Connect them here:

Now draw the pipeline, like the diagrams above, in the markdown cell below. For each arrow, label the **shape** of the data.

✍️ **My pipeline diagram:**

```text

```

## 🏁 Wrap-Up

You started this module with five separate `score` variables. You're finishing it with:

- **containers** that hold data in the right shape,
- **patterns** (Accumulate, Count, Filter, Best So Far) that process those containers,
- **functions** that give those patterns names, inputs, and outputs,
- **pipelines** that connect those functions.

That's the goal of the module:

> **Take a problem, decide how to store its data, break the work into functions, and pass data between them.**

**Next:** *Module 04 – More Functions & Loops* adds default values, keyword arguments, scope, `while` loops, and reading data from files, so your pipelines can start from a real file.

## 🔥 Challenge (Optional) — Grade Report Pipeline

Using the `students` list, build a pipeline that returns a dictionary counting how many students got each **letter grade** (A ≥ 90, B ≥ 80, C ≥ 70, D ≥ 60, F below that).

Suggested steps: `grades_of` → `to_letters` (list → list) → `frequency` (list → dict).

Expected: `{'A': 2, 'B': 1, 'D': 1, 'C': 1}`

In [ ]:
def to_letters(grades):
    pass

# Your pipeline here